<a href="https://colab.research.google.com/github/rithwik0908/llm-inference-benchmark/blob/main/notebooks/colab_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mistral FP16 → AWQ → vLLM benchmark



In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rithwik0908/llm-inference-benchmark.git"
REPO_COMMIT = "4b982710b03ca0d69f3c9a3c1516f93889d082f0"
ROOT = Path("/content/llm-inference-benchmark")
GPU_ENV = Path("/content/benchmark-venv")
GPU_PY = GPU_ENV / "bin/python"
assert "YOUR_HANDLE" not in REPO_URL, "Set your pushed repository URL first"
assert len(REPO_COMMIT) == 40 and all(c in "0123456789abcdef" for c in REPO_COMMIT)
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", REPO_COMMIT], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.26"], check=True)
subprocess.run(["uv", "venv", str(GPU_ENV), "--python", "3.11", "--seed"], check=True)
subprocess.run(
    ["uv", "pip", "sync", "--python", str(GPU_PY), "requirements/gpu.lock.txt"], check=True
)
subprocess.run(
    ["uv", "pip", "install", "--python", str(GPU_PY), "--no-deps", "-e", "."], check=True
)


def run(*args):
    subprocess.run([str(GPU_PY), *args], cwd=ROOT, check=True)


run("scripts/preflight.py", "--min-vram-gib", "22")

## 1. Freeze inputs and measure the FP16 baseline

No CPU offloading. Timings exclude weight loading and warmup. TTFT is first generated
token ID, not HTTP streaming TTFT. Prepared inputs use a fixed random-token workload;
the frozen MMLU questions are a separate quality workload.

In [ ]:
run("scripts/prepare_workload.py")
run("scripts/prepare_eval.py")
run("scripts/baseline_fp16.py")

## 2. Calibrate AWQ

The pinned llm-compressor implementation performs AWQ W4A16_ASYM, keeps `lm_head`
unquantized, and saves `compressed-tensors`. The source checkpoint is staged in host RAM;
use a high-RAM Colab runtime. Calibration uses the first 256 UltraChat training rows,
shuffled, at 512 tokens. It does not use MMLU questions. The process exits before serving.
A nonempty checkpoint directory is protected against accidental overwrite.

In [ ]:
run("scripts/quantize_awq.py")

## 3. Managed serving lifecycle

Processes avoid requiring Docker inside Colab. Each server writes a unique alias manifest.
The helper waits for health and checks early startup failure. Cleanup terminates the
server's process group so the next variant can reclaim VRAM. Inspect startup logs if
kernel selection, memory profiling, or startup fails. Do not bypass errors.

In [ ]:
import signal
import time
import urllib.error
import urllib.request

server = None
server_log = None


def stop_server():
    global server, server_log
    if server is not None:
        try:
            os.killpg(server.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(server.pid, signal.SIGKILL)
            server.wait(timeout=15)
        server = None
    if server_log:
        server_log.close()
        server_log = None


def start_server(variant):
    global server, server_log
    stop_server()
    server_log = (ROOT / f"results/server_{variant}.log").open("w")
    server = subprocess.Popen(
        [str(GPU_PY), "-m", "inference_bench.server", variant],
        cwd=ROOT,
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    deadline = time.monotonic() + 900
    try:
        while time.monotonic() < deadline:
            if server.poll() is not None:
                raise RuntimeError(f"Server exited; inspect results/server_{variant}.log")
            try:
                with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as response:
                    if response.status == 200:
                        return
            except (urllib.error.URLError, TimeoutError):
                pass
            time.sleep(5)
        raise TimeoutError("Server startup timed out; inspect startup log")
    except BaseException:
        stop_server()
        raise


def measure_variant(variant):
    try:
        start_server(variant)
        manifest = f"results/server_{variant}.json"
        run("scripts/evaluate_quality.py", "run", "--server-manifest", manifest)
        run("scripts/benchmark_serving.py", "--server-manifest", manifest)
    finally:
        stop_server()

## 4. Measure both variants, one at a time

Each sweep uses 512 requests at concurrency 1, 4, 16, 32 with three repetitions,
a fixed 256/128 token budget, continuous batching, and no prefix caching. Existing
sweep files are protected; archive previous measurements before rerunning.

In [ ]:
measure_variant("fp16")

In [ ]:
measure_variant("awq")

## 5. Compare and plot

The quality delta is AWQ minus FP16 in percentage points. With 50 questions, one
changed correct answer moves accuracy by 2 pp. A paired bootstrap interval expresses
some sampling uncertainty; this is not full MMLU or proof of equivalent quality.
Charts/report generation refuses failed sweeps and mismatched hardware/workloads.

In [ ]:
from IPython.display import Image, Markdown, display

run("scripts/evaluate_quality.py", "compare")
run("scripts/plot_results.py")

display(Markdown((ROOT / "results/REPORT.md").read_text()))
for name in ("throughput.png", "latency.png", "memory_quality.png"):
    display(Image(filename=str(ROOT / "results" / name)))

## 6. Preserve results and the AWQ checkpoint

Run this cell before disconnecting. It mounts your Drive and copies only project
artifacts, not credentials or the FP16 download cache. The checkpoint can be several GB.
Choose a new run directory each time. Copy `models/mistral-7b-awq/` to the laptop and
run `python scripts/edge_demo.py` there with the same pinned GPU environment.
That final step is deliberately not claimed as completed by this notebook.

In [ ]:
import shutil
from datetime import UTC, datetime

from google.colab import drive

drive.mount("/content/drive")
RUN_NAME = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
DEST = Path("/content/drive/MyDrive/llm-inference-benchmark") / RUN_NAME
DEST.mkdir(parents=True, exist_ok=False)
shutil.copytree(ROOT / "results", DEST / "results")
shutil.copytree(ROOT / "models/mistral-7b-awq", DEST / "models/mistral-7b-awq")
(DEST / "source_commit.txt").write_text(REPO_COMMIT + "\n")
print(f"Saved artifacts to {DEST}")

## Troubleshooting / honest limitations

- **OOM in baseline/quantization:** verify A100/L4 and high-RAM allocation; stop stale
  processes. Do not use CPU-offloaded inference as the baseline. Save logs and report
  failure if the requested allocation is unavailable.
- **Unsupported kernel:** record GPU/driver and complete startup traceback. The laptop
  sm_120 path needs its own integration check. No automatic precision/model fallback.
- **HTTP failure or missing stream usage:** the load test retains diagnostics and marks
  the run invalid. Do not include it in a before/after chart.
- **Changed GPU between variants:** rerun both on one GPU; the plotter rejects mixed runs.
- **Runtime lost:** restore checkpoint/results from Drive; do not mix partial sweeps from
  different GPUs. Complete runs are saved after each repetition, but partial sweeps
  intentionally cannot be plotted as complete benchmarks.
- **Need custom quality questions:** prepare once via `--custom-jsonl` before either
  quality run. Do not change the set between variants.
